In [1]:
# main.ipynb

# Install necessary packages if not already installed
# !pip install transformers chromadb sentence-transformers Levenshtein

# Import modules
import sys
sys.path.append('./')  # Adjust the path if necessary

from data_loader import load_user_reviews
from model_pipeline import RecommenderModel
from utils import extract_latest_n_reviews, extract_product_names, extract_ranked_products
from retrieval import initialize_chromadb, collect_results_per_product
from evaluation import normalize, compute_similarity, recall_at_k, ndcg_at_k

# Additional imports
import numpy as np
import torch
import pandas as pd

# Initialize Recommender Model
recommender_model = RecommenderModel()

# Initialize ChromaDB
db_path = "./chroma_db_mpnet"
db = initialize_chromadb(db_path)
collection_name = 'product_embeddings_filtered'  # Use the same collection name
collection = db.get_collection(name=collection_name)

# Load the product data
df = pd.read_csv("data/meta_all_beauty_filtered_simple.csv")

# Load training data
train_file = 'data/train_val_user_reviews.json'
input_set = load_user_reviews(train_file)

# Load test data
test_file = 'data/test_user_reviews.json'
input_set_test = load_user_reviews(test_file)

# Parameters
n_latest_reviews = 3
K = 10  # For Recall@K and NDCG@K
SIMILARITY_THRESHOLD = 90.0

# Initialize lists to store overall results
all_similarity_scores = []
all_matches = []
all_recalls = []
all_ndcgs = []

count = collection.count()
print(f"Total embeddings in collection: {count}")

c:\Users\Trung\anaconda3\envs\master_torch_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


max length is 4096
models/hf-frompretrained-download/meta-llama/Meta-Llama-3-8B-Instruct


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now set to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 2/2 [00:20<00:00, 10.17s/it]
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Total embeddings in collection: 356


In [5]:
# Parameters
n_latest_reviews = 3
SIMILARITY_THRESHOLD = 90.0
K = 10  # For Recall@K and NDCG@K
MAX_RETRIES = 3  # Maximum number of retries per user

# Initialize lists to collect metrics
all_similarity_scores = []
all_matches = []
all_recalls = []
all_ndcgs = []
skipped_users = []

with open('results.txt', 'w', encoding='utf-8') as result_file:
    # Loop over all users in the test set
    #num_users = 1  # Adjust as needed
    num_users = len(input_set)
    for userIndex in range(num_users):
        print(f"\nProcessing user {userIndex + 1}/{num_users}")

        retries = 0
        success = False

        while not success and retries < MAX_RETRIES:
            try:
                # Extract latest reviews for training
                example_user = [input_set[userIndex]]
                latest_reviews = extract_latest_n_reviews(example_user, n_latest_reviews)
                review_text = "\n".join([
                    f"Product: {rev['product_name']}\nReview: {rev['text']}"
                    for rev in latest_reviews
                ])

                # Check if latest_reviews is empty
                if not latest_reviews:
                    print(f"User {userIndex + 1} has no latest reviews. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped due to no latest reviews.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user

                # Create user profile
                profile = recommender_model.create_user_profile(review_text)
                print("\nGenerated User Profile:")
                print(profile)

                # Generate preliminary recommendations
                preliminary_rec = recommender_model.create_preliminary_recommendations(profile)
                print("\nPreliminary Recommendations:")
                print(preliminary_rec)

                # Extract product names
                product_names = extract_product_names(preliminary_rec)
                print("\nExtracted Product Names:")
                for idx, name in enumerate(product_names, 1):
                    print(f"{idx}. {name}")

                # Check if product_names is empty
                if not product_names:
                    print(f"User {userIndex + 1} has empty product names. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped due to empty product names.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user

                # Query ChromaDB and collect results
                final_results = collect_results_per_product(product_names, collection, max_products=20)
                if final_results == -1:
                    print(f"User {userIndex + 1} has no recommendations. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped due to no recommendations.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user
                print(f"Final results: {(final_results)}")     
                print(f"Final results: {len(final_results)} items")

                example_user_test = [input_set_test[userIndex]]
                test_review = extract_latest_n_reviews(example_user_test, 1)
                test_product = test_review[0]['parent_asin']  # Adjust field name if necessary
                print("\nTest Product:")
                print(test_product)

                recommended_products = []
                for doc,distance, metadata in final_results:
                    # Retrieve the product title using metadata (e.g., 'asin')
                    #print(final_results)
                    asin = metadata['metadata']
                    filt = df['parent_asin'] == asin
                    title = asin
                    print(asin)
                    if len(title) > 0:
                        recommended_products.append(title)
                    else:
                        recommended_products.append(doc)  # Fallback to the document if title not found

                # Proceed with evaluation if we have recommendations
                # Get the actual product from the test set

                print("normalizing")
                # Proceed with evaluation using ranked_product_asins
                normalized_test_product = normalize(test_product)
                normalized_ranked_products = [normalize(name) for name in recommended_products]
                print("computing similiarities")
                # Compute similarity scores and matches
                similarity_scores = []
                matches = []
                for rec_product in normalized_ranked_products:
                    sim_score = compute_similarity(rec_product, normalized_test_product)
                    similarity_scores.append(sim_score)
                    matches.append(sim_score >= SIMILARITY_THRESHOLD)

                # Display similarity scores and matches
                print("\nSimilarity Scores and Matches:")
                for idx, (asin, score, match) in enumerate(zip(recommended_products, similarity_scores, matches), 1):
                    print(f"{idx}. {asin}")
                    print(f"   Similarity Score: {score:.2f}%")
                    print(f"   Match: {'Yes' if match else 'No'}")

                # Save results for this user
                result_file.write(f"User {userIndex + 1}:\n")
                result_file.write(f"Test Product: {test_product}\n")
                result_file.write(f"Recommended Products:\n")
                for i, (asin, score, match) in enumerate(zip(recommended_products, similarity_scores, matches)):
                    result_file.write(f"  {i+1}. {asin} - Similarity: {score:.2f}% - {'Match' if match else 'No Match'}\n")
                result_file.write("\n")

                # Collect similarity scores and matches
                all_similarity_scores.extend(similarity_scores)
                all_matches.extend(matches)

                # Calculate Recall@K and NDCG@K for this user
                recall = recall_at_k(matches, K)
                ndcg = ndcg_at_k(matches, K)
                all_recalls.append(recall)
                all_ndcgs.append(ndcg)

                # Optional: Print progress
                print(f"User {userIndex + 1} - Recall@{K}: {recall}, NDCG@{K}: {ndcg}")

                # If everything succeeded, set success to True to exit the retry loop
                success = True

            except Exception as e:
                retries += 1
                print(f"User {userIndex + 1} encountered an error during processing: {e}")
                print(f"Retrying... ({retries}/{MAX_RETRIES})")
                if retries >= MAX_RETRIES:
                    print(f"User {userIndex + 1} exceeded maximum retries. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped after {MAX_RETRIES} retries due to errors.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user
                else:
                    print("Restarting user processing from the beginning.\n")
                    # The loop will restart, processing the user again

    # Calculate overall metrics after processing all users
    if all_recalls and all_ndcgs:
        mean_recall = np.mean(all_recalls)
        mean_ndcg = np.mean(all_ndcgs)
    else:
        mean_recall = 0.0
        mean_ndcg = 0.0

    # Write overall metrics and skipped users to the file
    result_file.write("Overall Evaluation Metrics:\n")
    result_file.write(f"Mean Recall@{K}: {mean_recall}\n")
    result_file.write(f"Mean NDCG@{K}: {mean_ndcg}\n\n")

    if skipped_users:
        result_file.write("Skipped Users:\n")
        result_file.write(", ".join(map(str, skipped_users)))
        result_file.write("\n")

# Print overall metrics
print("\nOverall Evaluation Metrics:")
print(f"Mean Recall@{K}: {mean_recall}")
print(f"Mean NDCG@{K}: {mean_ndcg}")



Processing user 1/1

Generated User Profile:
````
User Profile Analysis

**Long-term Preferences**

* Appreciation for high-quality products, evident through consistent positive feedback across various product categories.
* Value eco-friendliness, sustainability, and natural ingredients in personal care items.
* Emphasis on skincare routines, indicating importance placed on maintaining healthy-looking skin.
* Enjoyment of grooming kits and self-care activities suggests interest in pampering oneself.

**Short-term Interests**

* Recent exploration of new skincare practices, possibly driven by desire for more effective results.
* Interest in DIY keratin systems indicates curiosity around home beauty solutions.
* Possible transition towards vegan or cruelty-free lifestyles due to preference for animal-friendly products.

**Demographic Information**

* Age Range: Late twenties to early thirties; likely experiencing significant life changes, such as career advancement or family formation.


In [2]:
# Parameters
n_latest_reviews = 3
SIMILARITY_THRESHOLD = 90.0
K = 10  # For Recall@K and NDCG@K
MAX_RETRIES = 3  # Maximum number of retries per user

# Initialize lists to collect metrics
all_similarity_scores = []
all_matches = []
all_recalls = []
all_ndcgs = []
skipped_users = []

with open('results.txt', 'w', encoding='utf-8') as result_file:
    # Loop over all users in the test set
    num_users = 1  # Adjust as needed
    for userIndex in range(num_users):
        print(f"\nProcessing user {userIndex + 1}/{num_users}")

        retries = 0
        success = False

        while not success and retries < MAX_RETRIES:
            try:
                # Extract latest reviews for training
                example_user = [input_set[userIndex]]
                latest_reviews = extract_latest_n_reviews(example_user, n_latest_reviews)
                review_text = "\n".join([
                    f"Product: {rev['product_name']}\nReview: {rev['text']}"
                    for rev in latest_reviews
                ])

                # Check if latest_reviews is empty
                if not latest_reviews:
                    print(f"User {userIndex + 1} has no latest reviews. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped due to no latest reviews.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user

                # Create user profile
                profile = recommender_model.create_user_profile(review_text)
                print("\nGenerated User Profile:")
                print(profile)

                # Generate preliminary recommendations
                preliminary_rec = recommender_model.create_preliminary_recommendations(profile)
                print("\nPreliminary Recommendations:")
                print(preliminary_rec)

                # Extract product names
                product_names = extract_product_names(preliminary_rec)
                print("\nExtracted Product Names:")
                for idx, name in enumerate(product_names, 1):
                    print(f"{idx}. {name}")

                # Check if product_names is empty
                if not product_names:
                    print(f"User {userIndex + 1} has empty product names. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped due to empty product names.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user

                # Query ChromaDB and collect results
                final_results = collect_results_per_product(product_names, collection, max_products=20)
                if final_results == -1:
                    print(f"User {userIndex + 1} has no recommendations. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped due to no recommendations.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user
                print(f"Final results: {(final_results)}")     
                print(f"Final results: {len(final_results)} items")

                # Proceed with evaluation if we have recommendations
                # Get the actual product from the test set
                example_user_test = [input_set_test[userIndex]]
                test_review = extract_latest_n_reviews(example_user_test, 1)
                test_product = test_review[0]['parent_asin']  # Adjust field name if necessary
                print("\nTest Product:")
                print(test_product)

                # Retrieve recommended products (documents)
                products = [doc for doc, _, _ in final_results]

                # Prepare products text for the prompt
                products_text = "\n".join([f"- {product}" for product in products])

                # Call the ranker
                ranked_products_response = recommender_model.rank_recommendations(profile, products_text)
                print("\nRanked Products Response:")
                print(ranked_products_response)

                # Extract ranked product names
                ranked_products = extract_ranked_products(ranked_products_response)
                print("\nRanked Products:")
                for idx, product in enumerate(ranked_products, 1):
                    print(f"{idx}. {product}")

                # Map ranked products back to metadata
                # Create a mapping from product names to their metadata
                product_metadata_map = {doc: (distance, metadata) for (doc, distance, metadata) in final_results}

                # Retrieve the ASINs or identifiers for the ranked products
                ranked_product_asins = []
                for product in ranked_products:
                    # Attempt direct match
                    if product in product_metadata_map:
                        distance, metadata = product_metadata_map[product]
                        asin = metadata['metadata']
                        ranked_product_asins.append(asin)
                    else:
                        # Implement fuzzy matching or handle accordingly
                        print(f"Product '{product}' not found in original results. Skipping.")

                # Display recommended products with metadata
                print("\nRecommended Products:")
                for idx, asin in enumerate(ranked_product_asins, 1):
                    print(f"{idx}. {asin}")

                # Proceed with evaluation using ranked_product_asins
                normalized_test_product = normalize(test_product)
                normalized_ranked_products = [normalize(name) for name in final_results]

                # Compute similarity scores and matches
                similarity_scores = []
                matches = []
                for rec_product in normalized_ranked_products:
                    sim_score = compute_similarity(rec_product, normalized_test_product)
                    similarity_scores.append(sim_score)
                    matches.append(sim_score >= SIMILARITY_THRESHOLD)

                # Display similarity scores and matches
                print("\nSimilarity Scores and Matches:")
                for idx, (asin, score, match) in enumerate(zip(ranked_product_asins, similarity_scores, matches), 1):
                    print(f"{idx}. {asin}")
                    print(f"   Similarity Score: {score:.2f}%")
                    print(f"   Match: {'Yes' if match else 'No'}")

                # Save results for this user
                result_file.write(f"User {userIndex + 1}:\n")
                result_file.write(f"Test Product: {test_product}\n")
                result_file.write(f"Recommended Products:\n")
                for i, (asin, score, match) in enumerate(zip(ranked_product_asins, similarity_scores, matches)):
                    result_file.write(f"  {i+1}. {asin} - Similarity: {score:.2f}% - {'Match' if match else 'No Match'}\n")
                result_file.write("\n")

                # Collect similarity scores and matches
                all_similarity_scores.extend(similarity_scores)
                all_matches.extend(matches)

                # Calculate Recall@K and NDCG@K for this user
                recall = recall_at_k(matches, K)
                ndcg = ndcg_at_k(matches, K)
                all_recalls.append(recall)
                all_ndcgs.append(ndcg)

                # Optional: Print progress
                print(f"User {userIndex + 1} - Recall@{K}: {recall}, NDCG@{K}: {ndcg}")

                # If everything succeeded, set success to True to exit the retry loop
                success = True

            except Exception as e:
                retries += 1
                print(f"User {userIndex + 1} encountered an error during processing: {e}")
                print(f"Retrying... ({retries}/{MAX_RETRIES})")
                if retries >= MAX_RETRIES:
                    print(f"User {userIndex + 1} exceeded maximum retries. Skipping.")
                    result_file.write(f"User {userIndex + 1} skipped after {MAX_RETRIES} retries due to errors.\n\n")
                    skipped_users.append(userIndex + 1)
                    break  # Exit the retry loop and proceed to next user
                else:
                    print("Restarting user processing from the beginning.\n")
                    # The loop will restart, processing the user again

    # Calculate overall metrics after processing all users
    if all_recalls and all_ndcgs:
        mean_recall = np.mean(all_recalls)
        mean_ndcg = np.mean(all_ndcgs)
    else:
        mean_recall = 0.0
        mean_ndcg = 0.0

    # Write overall metrics and skipped users to the file
    result_file.write("Overall Evaluation Metrics:\n")
    result_file.write(f"Mean Recall@{K}: {mean_recall}\n")
    result_file.write(f"Mean NDCG@{K}: {mean_ndcg}\n\n")

    if skipped_users:
        result_file.write("Skipped Users:\n")
        result_file.write(", ".join(map(str, skipped_users)))
        result_file.write("\n")

# Print overall metrics
print("\nOverall Evaluation Metrics:")
print(f"Mean Recall@{K}: {mean_recall}")
print(f"Mean NDCG@{K}: {mean_ndcg}")



Processing user 1/1

Generated User Profile:
````
**Long-term Preferences**

* Appreciation for high-quality products
* Value sustainability and eco-friendliness
* Interest in natural skincare routines
* Preference for gentle, effective, and non-irritating formulas

**Short-term Interests**

* Recent interest in facial lotions and creams
* Exploration of DIY keratin systems
* Consideration of alternative grooming tools (e.g., bamboo cotton ear swabs)

**Demographic Information**

* Age range: Late 20s to early 40s (based on product usage and language style)
* Gender: Female (based on beauty-related purchases and self-care focus)
* Lifestyle: Busy professional with limited free time; values convenience, effectiveness, and environmental responsibility

**User Profile Summary**

This individual prioritizes quality, eco-friendly products and sustainable practices. They value effective yet gentle formulations and are willing to invest in premium items. Their short-term interests suggest ex